# 🔁 Day 5C — RAG Query Pipeline: Retrieve, Augment, Generate
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are

| Notebook | What we built |
|---|---|
| 5A | Embeddings + cosine similarity — the maths behind RAG |
| 5B | NumpyVectorStore — 20 tax chunks indexed and queryable |
| **5C** | **Wire retrieval into generation — the complete RAG pipeline** |

**This notebook completes Day 5.** By the end:
- `ask_with_rag()` — end-to-end: question → embed → retrieve → augment → cited answer
- `ask_without_rag()` — baseline using model training memory only
- Side-by-side comparison on the same questions
- Results exported to `day5_rag_results.json` for Day 6 reference

> **Self-contained:** this notebook rebuilds the vector store internally.
> You do not need Notebook 5B open in parallel.

---
## ⏱️ Time: 30 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai numpy --quiet

In [ ]:
import os
import re
import json
import numpy as np
import datetime
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI

In [ ]:
AZURE_OPENAI_ENDPOINT      = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY       = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME      = input("Chat deployment name (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT = input("Embeddings deployment name (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION          = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

---
## 🔧 Step 3: NumpyVectorStore (self-contained, no chromadb)

In [ ]:
class NumpyVectorStore:
    """
    Pure-Python in-memory vector store using numpy cosine similarity.
    API mirrors ChromaDB's .add() / .query() interface.
    Works in: Pyodide · Python 3.11 · 3.12 · 3.13
    """

    def __init__(self, name: str = "collection"):
        self.name       = name
        self.ids        = []
        self.documents  = []
        self.embeddings = []
        self.metadatas  = []

    def add(self, ids, documents, embeddings, metadatas):
        if isinstance(ids, str):
            ids, documents, embeddings, metadatas = [ids], [documents], [embeddings], [metadatas]
        self.ids.extend(ids)
        self.documents.extend(documents)
        self.embeddings.extend(embeddings)
        self.metadatas.extend(metadatas)

    def count(self) -> int:
        return len(self.ids)

    def query(self, query_embeddings, n_results: int = 3, where: dict = None):
        if not self.embeddings:
            raise ValueError("Vector store is empty.")
        q      = np.array(query_embeddings[0], dtype=np.float32)
        q_norm = q / (np.linalg.norm(q) + 1e-10)

        indices = list(range(len(self.ids)))
        if where:
            indices = [i for i in indices if all(
                self.metadatas[i].get(k) == v for k, v in where.items()
            )]

        if not indices:
            return {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

        matrix = np.array([self.embeddings[i] for i in indices], dtype=np.float32)
        norms  = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-10
        sims   = (matrix / norms) @ q_norm

        k       = min(n_results, len(indices))
        top_pos = np.argsort(sims)[::-1][:k]
        top_idx = [indices[p] for p in top_pos]

        return {
            "ids":       [[self.ids[i]       for i in top_idx]],
            "documents": [[self.documents[i] for i in top_idx]],
            "metadatas": [[self.metadatas[i] for i in top_idx]],
            "distances": [[round(float(1 - sims[p]), 4) for p in top_pos]],
        }


print("✅ NumpyVectorStore defined")

---
## 📚 Step 4: Knowledge Chunks + Embedding Helper

In [ ]:
TAX_KNOWLEDGE_CHUNKS = [
    {"id": "gst_export_001",
     "text": "Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 2017. A registered supplier may export without payment of IGST if a Letter of Undertaking (LUT) has been filed under Rule 96A. Alternatively, the supplier may export on payment of IGST and claim a refund under Section 54 of the CGST Act. LUT is valid for the entire financial year from the date of filing.",
     "metadata": {"source": "IGST Act 2017", "section": "16(1)", "type": "legislation", "topic": "exports"}},

    {"id": "gst_export_002",
     "text": "Place of supply for export of services is determined under Section 13 of the IGST Act. When the recipient is located outside India, the place of supply is the location of the recipient. IT/ITES services provided to overseas clients qualify as export of services, subject to: (a) supplier is in India, (b) recipient is outside India, (c) payment received in convertible foreign exchange, (d) supplier and recipient are not merely establishments of the same person.",
     "metadata": {"source": "IGST Act 2017", "section": "13", "type": "legislation", "topic": "exports"}},

    {"id": "gst_notification_01_2023",
     "text": "Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. Cloud computing services, SaaS subscriptions, and data processing services supplied by Indian entities to overseas recipients qualify as export of services and are zero-rated if LUT is filed. If LUT is not filed, IGST at 18% applies and refund may be claimed. Domestic B2B supply of these services attracts 18% GST.",
     "metadata": {"source": "Notification 01/2023-IT(Rate)", "section": "4", "type": "notification", "topic": "exports"}},

    {"id": "gst_composition_001",
     "text": "Section 10 of the CGST Act 2017 provides for the composition scheme. Registered suppliers with aggregate turnover not exceeding Rs 1.5 crore may opt for composition levy. Composition dealers pay tax at 1% for traders, 2% for manufacturers, and 5% for restaurants. Cannot collect tax from recipients, cannot issue tax invoices, and cannot avail input tax credit. Must issue a Bill of Supply instead.",
     "metadata": {"source": "CGST Act 2017", "section": "10", "type": "legislation", "topic": "composition_scheme"}},

    {"id": "gst_itc_001",
     "text": "Input Tax Credit (ITC) under Section 16 of the CGST Act can be claimed if: (a) the supplier has filed returns and paid the tax, (b) the recipient holds a valid tax invoice, (c) the goods or services have been received, and (d) the tax has been paid to the government. Section 17(5) lists blocked credits including motor vehicles for personal use, food and beverages, beauty treatment, health services, and club memberships. ITC cannot be claimed after filing of the annual return for the relevant financial year.",
     "metadata": {"source": "CGST Act 2017", "section": "16", "type": "legislation", "topic": "input_tax_credit"}},

    {"id": "gst_rcm_001",
     "text": "Reverse Charge Mechanism (RCM) under Section 9(3) of the CGST Act requires the recipient to pay GST instead of the supplier in notified categories. Key RCM notifications: legal services by an advocate (Notification 13/2017-CT), services by a director to a company, goods transport agency services, import of services from overseas suppliers, and security services by an individual to a body corporate.",
     "metadata": {"source": "CGST Act 2017", "section": "9(3)", "type": "legislation", "topic": "reverse_charge"}},

    {"id": "gst_registration_001",
     "text": "GST registration is mandatory when aggregate turnover exceeds Rs 20 lakh for service providers (Rs 10 lakh in special category states) and Rs 40 lakh for goods suppliers. Inter-state suppliers must register regardless of turnover. E-commerce operators must also register regardless of turnover. Voluntary registration is permitted below the threshold.",
     "metadata": {"source": "CGST Act 2017", "section": "22", "type": "legislation", "topic": "registration"}},

    {"id": "gst_gstr1_001",
     "text": "GSTR-1 is the return for outward supplies. Monthly filers with turnover above Rs 5 crore must file by the 11th of the following month. Quarterly filers under QRMP scheme file by the 13th of the month following each quarter. Late filing attracts Rs 50/day (Rs 20/day for nil returns).",
     "metadata": {"source": "CGST Rules 2017", "section": "Rule 59", "type": "rule", "topic": "filing"}},

    {"id": "gst_gstr3b_001",
     "text": "GSTR-3B is the monthly summary return for payment of GST liability. Monthly taxpayers file by the 20th of the following month. Under QRMP scheme, GSTR-3B is filed quarterly by the 22nd for Category-I states and 24th for Category-II states. Payment must accompany filing.",
     "metadata": {"source": "CGST Rules 2017", "section": "Rule 61", "type": "rule", "topic": "filing"}},

    {"id": "tds_194c_001",
     "text": "Section 194C requires TDS on payments to contractors and sub-contractors. TDS rate is 1% for individuals and HUFs, and 2% for companies and firms. TDS is deducted when a single payment exceeds Rs 30,000 or aggregate payments in a financial year exceed Rs 1,00,000. Also covers advertising contracts, catering, broadcasting, and carriage of goods.",
     "metadata": {"source": "Income Tax Act 1961", "section": "194C", "type": "legislation", "topic": "tds"}},

    {"id": "tds_194j_001",
     "text": "Section 194J covers TDS on fees for professional or technical services. TDS rate is 10% for professional services (lawyers, doctors, engineers, chartered accountants) and 2% for technical services. The threshold is Rs 30,000 per annum per payee. Technical services include managerial and consultancy services not covered under Section 194C.",
     "metadata": {"source": "Income Tax Act 1961", "section": "194J", "type": "legislation", "topic": "tds"}},

    {"id": "tds_return_001",
     "text": "TDS returns must be filed quarterly. Form 24Q for TDS on salaries, Form 26Q for non-salary payments to residents, Form 27Q for payments to non-residents. Due dates: Q1 July 31; Q2 October 31; Q3 January 31; Q4 May 31. TDS certificates Form 16 and Form 16A must be issued within 15 days of the due date of filing.",
     "metadata": {"source": "Income Tax Rules 1962", "section": "Rule 31A", "type": "rule", "topic": "tds"}},

    {"id": "corp_tax_rates_001",
     "text": "Corporate income tax rates for Indian domestic companies: 25% for companies with turnover not exceeding Rs 400 crore; 30% for others. Section 115BAB provides 15% for new domestic manufacturing companies incorporated after October 1, 2019. Effective rate including surcharge and cess is approximately 25.17% for the 25% slab.",
     "metadata": {"source": "Income Tax Act 1961", "section": "115BAB", "type": "legislation", "topic": "corporate_tax"}},

    {"id": "transfer_pricing_001",
     "text": "Transfer pricing under Sections 92-92F requires international transactions between associated enterprises to be at arm's length price. Methods: CUP, resale price, cost plus, TNMM, and profit split. Taxpayers with international transactions exceeding Rs 1 crore must obtain a Transfer Pricing Report in Form 3CEB from a CA. Due date: October 31.",
     "metadata": {"source": "Income Tax Act 1961", "section": "92-92F", "type": "legislation", "topic": "transfer_pricing"}},

    {"id": "advance_tax_001",
     "text": "Advance tax is payable when estimated tax liability exceeds Rs 10,000. Instalments: 15% by June 15, 45% cumulative by September 15, 75% cumulative by December 15, 100% by March 15. Interest under Section 234B applies at 1% per month for default. Senior citizens with no business income are exempt.",
     "metadata": {"source": "Income Tax Act 1961", "section": "208-219", "type": "legislation", "topic": "advance_tax"}},

    {"id": "section_80c_001",
     "text": "Section 80C allows deductions up to Rs 1,50,000 per year. Eligible: LIC premiums, PPF, ELSS, NSC, 5-year FDs, tuition fees for two children, EPF, and home loan principal repayment. Available only under the old tax regime. Under the new regime (Section 115BAC), Section 80C deductions are not available.",
     "metadata": {"source": "Income Tax Act 1961", "section": "80C", "type": "legislation", "topic": "deductions"}},

    {"id": "new_tax_regime_001",
     "text": "New tax regime under Section 115BAC (FY 2025-26): Nil up to Rs 3 lakh, 5% for Rs 3-7 lakh, 10% for Rs 7-10 lakh, 15% for Rs 10-12 lakh, 20% for Rs 12-15 lakh, 30% above Rs 15 lakh. Standard deduction Rs 75,000. Rebate under Section 87A: zero tax up to Rs 7 lakh income. No 80C or HRA deductions. Default regime from AY 2024-25.",
     "metadata": {"source": "Income Tax Act 1961", "section": "115BAC", "type": "legislation", "topic": "income_tax_slabs"}},

    {"id": "ltcg_001",
     "text": "LTCG on listed equity and equity-oriented mutual funds exceeding Rs 1 lakh is taxed at 10% without indexation under Section 112A. Holding period: more than 12 months for equity; more than 24 months for unlisted shares and immovable property; more than 36 months for debt mutual funds. Short-term capital gains on listed equity taxed at 15% under Section 111A.",
     "metadata": {"source": "Income Tax Act 1961", "section": "112A", "type": "legislation", "topic": "capital_gains"}},

    {"id": "itr_filing_001",
     "text": "Income tax return filing due dates: July 31 for individuals and HUFs; October 31 for tax audit cases and companies; November 30 for transfer pricing cases. Belated return by December 31 with late fee under Section 234F. Revised return also by December 31 of the assessment year.",
     "metadata": {"source": "Income Tax Act 1961", "section": "139", "type": "legislation", "topic": "filing"}},

    {"id": "gst_works_contract_001",
     "text": "Works contract under GST is treated as a supply of service. GST rates: 18% general; 12% for construction of affordable housing, government civil structures, and roads. ITC on works contracts is allowed to the contractor, subject to Section 17(5)(c) restrictions on immovable property construction.",
     "metadata": {"source": "Notification 11/2017-CT(Rate)", "section": "Entry 3", "type": "notification", "topic": "works_contract"}},
]


def get_embedding(text: str) -> list:
    response = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = text
    )
    return response.data[0].embedding


print(f"✅ {len(TAX_KNOWLEDGE_CHUNKS)} chunks defined")

---
## 🏗️ Step 5: Build the Vector Store

In [ ]:
tax_collection = NumpyVectorStore(name="tax_knowledge")

print(f"Embedding {len(TAX_KNOWLEDGE_CHUNKS)} chunks...")
for i, chunk in enumerate(TAX_KNOWLEDGE_CHUNKS, 1):
    vec = get_embedding(chunk["text"])
    tax_collection.add(
        ids        = [chunk["id"]],
        documents  = [chunk["text"]],
        embeddings = [vec],
        metadatas  = [{k: str(v) for k, v in chunk["metadata"].items()}],
    )
    print(f"  [{i:02d}/{len(TAX_KNOWLEDGE_CHUNKS)}]", end="\r")

print(f"\n✅ Vector store ready — {tax_collection.count()} chunks indexed")

---
## 🔧 Step 6: Build the RAG Pipeline Functions

In [ ]:
# ── System prompts ────────────────────────────────────────────────────────────
RAG_SYSTEM_PROMPT = (
    "You are a senior tax consultant at a Big4 firm.\n"
    "Answer questions using ONLY the context provided below. Do not use training memory.\n\n"
    "RULES:\n"
    "1. Every factual statement must cite its source in square brackets, "
    "e.g. [IGST Act S.16(1)] or [Notification 01/2023-IT(Rate) S.4].\n"
    "2. If the answer is not in the provided context, say: "
    "'This specific question is not covered in the retrieved sections. "
    "Please consult the full legislation.'\n"
    "3. Keep answers concise: 3-6 sentences maximum."
)

BASELINE_SYSTEM_PROMPT = (
    "You are a senior tax consultant at a Big4 firm. "
    "Answer Indian tax questions accurately and concisely in 3-6 sentences."
)


# ── Retrieval ─────────────────────────────────────────────────────────────────
def retrieve(query: str, n_results: int = 3) -> list:
    """Embed the query and return top-k chunks with similarity scores."""
    q_vec   = get_embedding(query)
    results = tax_collection.query(query_embeddings=[q_vec], n_results=n_results)
    return [
        {
            "id":         results["ids"][0][i],
            "text":       results["documents"][0][i],
            "metadata":   results["metadatas"][0][i],
            "similarity": round(1 - results["distances"][0][i], 3),
        }
        for i in range(len(results["ids"][0]))
    ]


def format_context(chunks: list) -> str:
    """Format retrieved chunks as a numbered context block."""
    lines = ["RETRIEVED CONTEXT:\n"]
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        lines.append(f"[Source {i}: {m.get('source','?')}  §{m.get('section','?')}]")
        lines.append(c["text"])
        lines.append("")
    return "\n".join(lines)


# ── Generation: baseline (no RAG) ────────────────────────────────────────────
def ask_without_rag(question: str) -> str:
    """Answer using only model training knowledge — no retrieval."""
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 400,
    )
    return response.choices[0].message.content


# ── Generation: with RAG ──────────────────────────────────────────────────────
def ask_with_rag(question: str, n_chunks: int = 3, verbose: bool = False):
    """
    Full RAG pipeline:
      1. Embed the question
      2. Retrieve top-k chunks from NumpyVectorStore
      3. Inject chunks as context into the system prompt
      4. Generate a grounded, cited answer via Azure OpenAI

    Returns: (answer_str, chunks_list)
    """
    # Steps 1 + 2
    chunks = retrieve(question, n_results=n_chunks)

    if verbose:
        print("  Retrieved chunks:")
        for c in chunks:
            print(f"    [{c['similarity']:.3f}] {c['id']}")

    # Step 3
    context          = format_context(chunks)
    augmented_system = RAG_SYSTEM_PROMPT + "\n\n" + context

    # Step 4
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": augmented_system},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 500,
    )
    return response.choices[0].message.content, chunks


print("✅ ask_without_rag() and ask_with_rag() ready")

---
## 🧪 Step 7: Test on 5 Tax Questions

In [ ]:
TEST_QUESTIONS = [
    "What GST rate applies to cloud software exported to a US company with LUT filed?",
    "A company avails legal services from a senior advocate. Does GST apply and who pays?",
    "What are the income tax slabs under the new tax regime for FY 2025-26?",
    "When is GSTR-3B due for a monthly taxpayer? What does it cover?",
    "Can I claim ITC on food expenses incurred for office employees under GST?",
]

rag_results = []

for q in TEST_QUESTIONS:
    print(f"\n{'─'*65}")
    print(f"Q: {q}")
    print(f"{'─'*65}")
    answer, chunks = ask_with_rag(q, n_chunks=3, verbose=True)
    print(f"\nA: {answer}")
    rag_results.append({
        "question":    q,
        "answer":      answer,
        "chunks_used": [c["id"] for c in chunks]
    })

---
## 🆚 Step 8: Side-by-Side — Without RAG vs With RAG

In [ ]:
COMPARISON_QUESTION = "What GST rate applies to cloud software exported to a US company with LUT filed?"

print("\n" + "="*70)
print(f"QUESTION: {COMPARISON_QUESTION}")
print("="*70)

print("\n--- WITHOUT RAG (model training memory only) ---")
ans_baseline = ask_without_rag(COMPARISON_QUESTION)
print(ans_baseline)

print("\n--- WITH RAG (retrieved from vector store) ---")
ans_rag, chunks_used = ask_with_rag(COMPARISON_QUESTION, verbose=True)
print(ans_rag)

print("\n" + "="*70)
print("ANALYSIS")
print("="*70)
citation_patterns   = [r"\[.*?\]", r"Section \d+", r"Notification", r"Rule \d+"]
rag_has_citation    = any(re.search(p, ans_rag)      for p in citation_patterns)
base_has_citation   = any(re.search(p, ans_baseline) for p in citation_patterns)
print(f"  Without RAG — contains citation : {base_has_citation}")
print(f"  With RAG    — contains citation : {rag_has_citation}")
print()
print("  Chunks retrieved for the RAG answer:")
for c in chunks_used:
    print(f"    [{c['similarity']:.3f}]  {c['id']}  —  {c['metadata'].get('source','?')} §{c['metadata'].get('section','?')}")

---
## 📊 Step 9: Citation Quality Audit — All 5 Questions

In [ ]:
citation_patterns = [r"\[.*?\]", r"Section \d+", r"Notification", r"Rule \d+"]

print("CITATION QUALITY AUDIT")
print(f"{'Q#':<5} {'Has citation':>14} {'Chunks used':>13}")
print("-" * 35)

for i, r in enumerate(rag_results, 1):
    has_cit = any(re.search(p, r["answer"]) for p in citation_patterns)
    print(f"  Q{i:<3} {'YES' if has_cit else 'NO':>13}  {len(r['chunks_used']):>11}")

all_cite = sum(1 for r in rag_results if any(re.search(p, r["answer"]) for p in citation_patterns))
print()
print(f"  Answers with citations: {all_cite}/{len(rag_results)}")
print()
print("Any answer that cannot cite a source section should trigger human review.")
print("Day 6 formalises this into an evaluation metric: 'faithfulness score'.")

---
## 💾 Step 10: Export Results

In [ ]:
output = {
    "notebook":           "Day5C_RAG_Pipeline",
    "exported_at":        datetime.datetime.now().isoformat(),
    "model":              AZURE_DEPLOYMENT_NAME,
    "embedding_model":    AZURE_EMBEDDING_DEPLOYMENT,
    "vector_store":       "NumpyVectorStore",
    "knowledge_base_size":tax_collection.count(),
    "results":            rag_results,
    "comparison": {
        "question":    COMPARISON_QUESTION,
        "without_rag": ans_baseline,
        "with_rag":    ans_rag,
        "chunks_used": [c["id"] for c in chunks_used],
    }
}

with open("day5_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✅ Results saved to day5_rag_results.json")
print(f"   Questions answered : {len(rag_results)}")
print(f"   Chunks in store    : {tax_collection.count()}")

---
## 🎓 The Full Day 5 Arc

```
Day 2  →  ReAct agent: tool calls + short-term memory
Day 3  →  CoT + few-shot: structured outputs from clean questions
Day 4  →  Function calling: extraction from unstructured documents
Day 5  →  RAG: agent finds the right document before answering

5A:  embeddings — meaning encoded as numbers
5B:  NumpyVectorStore — 20 tax chunks indexed and queryable
5C:  RAG pipeline — retrieve → augment → generate with citations
```

---
## ➡️ Day 6: Advanced RAG & Source Citation

Day 5 retrieves. Day 6 makes retrieval production-grade:
- Extract page numbers and section numbers from metadata into formatted references
- Multi-document retrieval with deduplication
- RAG evaluation: relevance, faithfulness, and answer correctness scores
- Test against a golden Q&A dataset to measure retrieval quality numerically